In [ ]:
from pathlib import Path
from tempfile import NamedTemporaryFile

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

DATA_DIR = Path("../datapoints")
OUTPUT_DIR = DATA_DIR.parent / "datapoints_preprocessed"
WRITE_OUTPUTS = True

SYMBOLS = [
    "XAU", "USD", "QQQ.US", "MSFT.US", "HPE.US", "GOOGL.US", "GBP", "EUR",
    "ETH", "E_SANDP-500", "E_NQ-100", "E_D&J-IND", "BTC", "AMZN.US", "AMD.US", "AAPL.US",
]
TIMEFRAMES = ["15m"]
BAR_INTERVALS = {
    "1m": pd.Timedelta(minutes=1),
    "5m": pd.Timedelta(minutes=5),
    "15m": pd.Timedelta(minutes=15),
    "30m": pd.Timedelta(minutes=30),
    "1h": pd.Timedelta(hours=1),
    "4h": pd.Timedelta(hours=4),
}
# Use "close" if the CSV timestamp is the end of the candle.
BAR_TIMESTAMP = "open"
DATA_AS_OF_UTC = pd.Timestamp.now(tz="UTC")

## 1. Load and clean the candles

Remove bad timestamps and prices, keep missing volume as `NaN`, and check for duplicates and gaps.

In [ ]:
PRICE_COLUMNS = ["open", "high", "low", "close"]
RAW_COLUMNS = ["time", *PRICE_COLUMNS, "volume"]
SOURCE_COLUMNS = [*RAW_COLUMNS, "volume_delta"]


def load_clean_bars(file_path, interval, as_of_utc):
    df = pd.read_csv(file_path, usecols=lambda column: column in SOURCE_COLUMNS)
    missing_columns = sorted(set(RAW_COLUMNS) - set(df.columns))
    if missing_columns:
        raise ValueError(f"{file_path}: missing required columns {missing_columns}")

    df = df.rename(columns={"volume_delta": "source_volume_delta"})
    # Read timestamps without a timezone as UTC.
    df["time"] = pd.to_datetime(df["time"], format="mixed", utc=True, errors="coerce")
    invalid_time = df["time"].isna()
    audit = {"input_rows": len(df), "invalid_time_rows": int(invalid_time.sum())}
    df = df.loc[~invalid_time].copy()

    for column in [*PRICE_COLUMNS, "volume"]:
        df[column] = pd.to_numeric(df[column], errors="coerce").astype(float)
    if "source_volume_delta" in df:
        df["source_volume_delta"] = pd.to_numeric(df["source_volume_delta"], errors="coerce")

    prices = df[PRICE_COLUMNS]
    valid_prices = np.isfinite(prices).all(axis=1) & prices.gt(0).all(axis=1)
    del prices
    valid_prices &= df["high"].ge(df[["open", "close", "low"]].max(axis=1))
    valid_prices &= df["low"].le(df[["open", "close"]].min(axis=1))
    audit["invalid_price_rows"] = int((~valid_prices).sum())
    df = df.loc[valid_prices].copy()
    df["volume"] = df["volume"].where(np.isfinite(df["volume"]) & df["volume"].ge(0))

    # A candle can be used once it closes.
    df["available_at"] = df["time"]
    if BAR_TIMESTAMP == "open":
        df["available_at"] = df["time"] + interval
    incomplete = df["available_at"].gt(as_of_utc)
    audit["incomplete_or_future_rows"] = int(incomplete.sum())
    df = df.loc[~incomplete].copy()

    # Drop identical candles and stop if values at the same time disagree.
    rows_before = len(df)
    df = df.drop_duplicates(subset=RAW_COLUMNS)
    audit["identical_ohlcv_duplicates_removed"] = rows_before - len(df)
    duplicate_times = df.loc[df["time"].duplicated(), "time"]
    if not duplicate_times.empty:
        examples = duplicate_times.drop_duplicates().head(3).tolist()
        raise ValueError(f"{file_path}: conflicting OHLCV rows at {examples}")

    df = df.sort_values("time").reset_index(drop=True)
    time_gap = df["time"].diff()
    consecutive = time_gap.eq(interval)
    df["time_gap"] = time_gap
    df["gap_before"] = time_gap.notna() & ~consecutive
    observed_return = df["close"].pct_change(fill_method=None)
    df["return_1"] = observed_return.where(consecutive)
    df["price_jump"] = observed_return.abs().gt(0.10)
    audit.update(
        clean_rows=len(df),
        irregular_intervals=int(df["gap_before"].sum()),
        price_jumps_over_10pct=int(df["price_jump"].sum()),
        missing_or_invalid_volume_rows=int(df["volume"].isna().sum()),
        zero_volume_rows=int(df["volume"].eq(0).sum()),
    )
    return df, audit

## 2. Add indicators

Use the current candle and earlier candles. Ichimoku’s Chikou is left out because it uses future closes. Relative volume and CCI stay blank when their denominator is zero.

In [ ]:
# Moving averages and Bollinger Bands
def add_sma(df, period=20, column='close'):
    df[f'sma_{period}'] = df[column].rolling(window=period, min_periods=period).mean()
    return df


def add_ema(df, period=20, column='close'):
    values = df[column].to_numpy(dtype=float)
    ema = np.full(len(values), np.nan)
    if len(values) >= period:
        k = 2 / (period + 1)
        ema[period - 1] = values[:period].mean()
        for i in range(period, len(values)):
            ema[i] = values[i] * k + ema[i - 1] * (1 - k)
    df[f'ema_{period}'] = ema
    return df


def add_dma(df, period=20, displacement=5, column='close'):
    sma = df[column].rolling(window=period, min_periods=period).mean()
    df[f'dma_{period}_{displacement}'] = sma.shift(displacement)
    return df


def add_bollinger_bands(df, period=20, num_std=2, column='close'):
    rolling = df[column].rolling(window=period, min_periods=period)
    middle = rolling.mean()
    std = rolling.std(ddof=0)
    df['bb_middle'] = middle
    df['bb_upper'] = middle + num_std * std
    df['bb_lower'] = middle - num_std * std
    return df

In [ ]:
# Momentum indicators
def add_rsi(df, period=14, column='close'):
    change = df[column].diff().to_numpy(dtype=float)
    gain = np.where(change > 0, change, 0.0)
    loss = np.where(change < 0, -change, 0.0)
    rsi = np.full(len(df), np.nan)
    if len(df) > period:
        avg_gain = gain[1:period + 1].mean()
        avg_loss = loss[1:period + 1].mean()
        for i in range(period, len(df)):
            if i > period:
                avg_gain = (avg_gain * (period - 1) + gain[i]) / period
                avg_loss = (avg_loss * (period - 1) + loss[i]) / period
            if avg_loss == 0:
                rsi[i] = 100.0 if avg_gain > 0 else 50.0
            else:
                rsi[i] = 100 - 100 / (1 + avg_gain / avg_loss)
    df[f'rsi_{period}'] = rsi
    return df


def add_momentum(df, period=10, column='close'):
    df[f'momentum_{period}'] = df[column] - df[column].shift(period)
    return df


def add_macd(df, fast=12, slow=26, signal=9, column='close'):
    fast_ema = df[column].ewm(span=fast, adjust=False, min_periods=fast).mean()
    slow_ema = df[column].ewm(span=slow, adjust=False, min_periods=slow).mean()
    macd = fast_ema - slow_ema
    macd_signal = macd.ewm(span=signal, adjust=False, min_periods=signal).mean()
    df['macd'] = macd
    df['macd_signal'] = macd_signal
    df['macd_histogram'] = macd - macd_signal
    return df

In [ ]:
# Volume
def add_volume(df, period=20):
    df[f'volume_ma_{period}'] = df['volume'].rolling(window=period, min_periods=period).mean()
    volume_mean = df[f'volume_ma_{period}']
    df[f'relative_volume_{period}'] = df['volume'] / volume_mean.where(volume_mean.gt(0))
    return df


def add_volume_delta_estimate(df):
    candle_range = (df['high'] - df['low']).replace(0, np.nan)
    body_ratio = ((df['close'] - df['open']) / candle_range).fillna(0.0)
    df['volume_delta_estimate'] = df['volume'] * body_ratio
    return df

In [ ]:
# Ichimoku and ATR
def add_ichimoku(df, tenkan=9, kijun=26, senkou_b=52, displacement=26):
    def midpoint(period):
        high = df['high'].rolling(window=period, min_periods=period).max()
        low = df['low'].rolling(window=period, min_periods=period).min()
        return (high + low) / 2

    tenkan_sen = midpoint(tenkan)
    kijun_sen = midpoint(kijun)
    df['ichimoku_tenkan'] = tenkan_sen
    df['ichimoku_kijun'] = kijun_sen
    df['ichimoku_senkou_a'] = ((tenkan_sen + kijun_sen) / 2).shift(displacement)
    df['ichimoku_senkou_b'] = midpoint(senkou_b).shift(displacement)
    return df


def add_atr(df, period=14, method='wilder'):
    prev_close = df['close'].shift(1)
    ranges = [
        df['high'] - df['low'],
        (df['high'] - prev_close).abs(),
        (df['low'] - prev_close).abs(),
    ]
    true_range = pd.concat(ranges, axis=1).max(axis=1).to_numpy(dtype=float)

    if method == 'sma':
        atr = pd.Series(true_range).rolling(window=period, min_periods=period).mean().to_numpy()
        name = f'atr_sma_{period}'
    elif method == 'wilder':
        atr = np.full(len(true_range), np.nan)
        if len(true_range) >= period:
            atr[period - 1] = true_range[:period].mean()
            for i in range(period, len(true_range)):
                atr[i] = (atr[i - 1] * (period - 1) + true_range[i]) / period
        name = f'atr_{period}'
    else:
        raise ValueError("method must be 'wilder' or 'sma'")

    df['true_range'] = true_range
    df[name] = atr
    return df

In [ ]:
# Candle body and CCI
def add_body_pct(df):
    """Candle body as a fraction of the open. A value of 0.01 means +1%."""
    open_price = df['open'].replace(0, np.nan)
    df['body_pct'] = (df['close'] - df['open']) / open_price
    return df


def cci(df, window=14):
    typical_price = (df['high'] + df['low'] + df['close']) / 3
    average = typical_price.rolling(window).mean()
    mean_deviation = pd.Series(np.nan, index=df.index)

    if len(df) >= window:
        windows = sliding_window_view(typical_price.to_numpy(), window)
        # Calculate several windows at once and limit memory on large files.
        batch_size = 50_000
        for start in range(0, len(windows), batch_size):
            batch = windows[start:start + batch_size]
            center = batch.mean(axis=1, keepdims=True)
            first_row = start + window - 1
            mean_deviation.iloc[first_row:first_row + len(batch)] = (
                np.abs(batch - center).mean(axis=1)
            )

    df[f'cci_{window}'] = (typical_price - average) / (0.015 * mean_deviation.replace(0, np.nan))
    return df

In [ ]:
BASELINE_COLUMNS = ["return_1", "body_pct"]
INDICATOR_COLUMNS = [
    "sma_20", "ema_20", "dma_20_5", "bb_middle", "bb_upper", "bb_lower",
    "rsi_14", "momentum_10", "macd", "macd_signal", "macd_histogram",
    "volume_ma_20", "relative_volume_20", "volume_delta_estimate",
    "ichimoku_tenkan", "ichimoku_kijun", "ichimoku_senkou_a", "ichimoku_senkou_b",
    "true_range", "atr_sma_14", "cci_14", "cci_20",
]
FEATURE_COLUMNS = [*BASELINE_COLUMNS, *INDICATOR_COLUMNS]
MAX_WARMUP_ROWS = 77  # Senkou B uses 52 bars and a 26-bar shift.


# Add new indicators here and list their output columns above.
def add_indicators(df):
    add_body_pct(df)
    add_sma(df, period=20)
    add_ema(df, period=20)
    add_dma(df, period=20, displacement=5)
    add_bollinger_bands(df, period=20)
    add_rsi(df, period=14)
    add_momentum(df, period=10)
    add_macd(df)
    add_volume(df, period=20)
    add_volume_delta_estimate(df)
    add_ichimoku(df)
    add_atr(df, period=14, method="sma")
    cci(df, window=14)
    cci(df, window=20)
    df[FEATURE_COLUMNS] = df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    return df

## 3. Label the next candle

`target_up` is 1 if the next close is higher and 0 if it is lower or the same. `target_log_return` measures the price change. Leave both labels blank if the next candle is missing or there is a gap.

In [ ]:
def add_next_bar_targets(df, interval):
    next_bar_time = df["available_at"].shift(-1)
    has_next_bar = (next_bar_time - df["available_at"]).eq(interval)
    next_close = df["close"].shift(-1).where(has_next_bar)

    df["target_end"] = next_bar_time.where(has_next_bar)
    df["target_log_return"] = np.log(next_close) - np.log(df["close"])
    df["target_up"] = next_close.gt(df["close"]).where(has_next_bar).astype("Int8")
    return df

## 4. Process the CSVs

Run the last cell to see a summary, missing features after warm-up, and a few example rows. Set `WRITE_OUTPUTS = True` to save the results in `OUTPUT_DIR`.

In [ ]:
def run_preprocessing():
    if BAR_TIMESTAMP not in {"open", "close"}:
        raise ValueError("BAR_TIMESTAMP must be 'open' or 'close'.")
    unsupported = sorted(set(TIMEFRAMES) - set(BAR_INTERVALS))
    if unsupported:
        raise ValueError(f"Unsupported timeframes: {unsupported}")
    if DATA_AS_OF_UTC.tzinfo is None:
        raise ValueError("DATA_AS_OF_UTC must include a timezone.")
    if not DATA_DIR.is_dir():
        raise FileNotFoundError(f"Set DATA_DIR to the candle CSV directory. Checked {DATA_DIR.resolve()}")
    if WRITE_OUTPUTS and OUTPUT_DIR.resolve() == DATA_DIR.resolve():
        raise ValueError("Use a separate output directory to protect the source CSVs.")

    summaries = []
    missingness = []
    preview = None

    for symbol in SYMBOLS:
        for timeframe in TIMEFRAMES:
            identity = {"symbol": symbol, "timeframe": timeframe}
            source_path = DATA_DIR / symbol / f"{symbol}_{timeframe}.csv"
            if not source_path.is_file():
                summaries.append({**identity, "status": "file missing"})
                continue

            print(f"Processing {symbol} {timeframe}", flush=True)
            interval = BAR_INTERVALS[timeframe]
            df, audit = load_clean_bars(source_path, interval, DATA_AS_OF_UTC)
            df = add_indicators(df)
            df = add_next_bar_targets(df, interval)
            df.insert(0, "symbol", symbol)
            df.insert(1, "timeframe", timeframe)

            features = df[FEATURE_COLUMNS]
            labeled = df["target_end"].notna()
            summaries.append({
                **identity,
                "status": "processed",
                **audit,
                "labeled_rows": int(labeled.sum()),
                "flat_target_rows": int(df["target_log_return"].eq(0).sum()),
                "rows_with_all_features_and_label": int(
                    (labeled & features.notna().all(axis=1)).sum()
                ),
            })
            after_warmup = features.iloc[MAX_WARMUP_ROWS:]
            missing_total = features.isna().sum()
            missing_after_warmup = after_warmup.isna().sum()
            for feature in FEATURE_COLUMNS:
                missingness.append({
                    **identity,
                    "feature": feature,
                    "missing_total": int(missing_total[feature]),
                    "rows_after_warmup": len(after_warmup),
                    "missing_after_warmup": int(missing_after_warmup[feature]),
                })

            del features, after_warmup

            if preview is None and not df.empty:
                preview = df[[
                    "symbol", "timeframe", "available_at", "close", "relative_volume_20",
                    "atr_sma_14", "cci_14", "cci_20", "target_end", "target_up",
                ]].tail(8).copy()

            if WRITE_OUTPUTS:
                destination = OUTPUT_DIR / symbol / f"{symbol}_{timeframe}_features.csv"
                destination.parent.mkdir(parents=True, exist_ok=True)
                # Replace the CSV only after the new file is complete.
                with NamedTemporaryFile(dir=destination.parent, suffix=".tmp", delete=False) as handle:
                    temporary_path = Path(handle.name)
                try:
                    df.to_csv(temporary_path, index=False)
                    temporary_path.replace(destination)
                finally:
                    temporary_path.unlink(missing_ok=True)

            del df

    if not missingness:
        print("No matching CSVs were processed. Check DATA_DIR, SYMBOLS and TIMEFRAMES.")
    return pd.DataFrame(summaries), pd.DataFrame(missingness), preview

In [ ]:
quality_summary, feature_audit, preview = run_preprocessing()
display(quality_summary)
if not feature_audit.empty:
    missing_features = feature_audit["missing_after_warmup"] > 0
    display(feature_audit.loc[missing_features].head(30))
if preview is not None:
    display(preview)